# Nine Dimensionality-Reduction Methods

Complete companion to the original guide: PCA and its full/randomized/incremental/kernel variants, LLE, t-SNE, LDA and MDS. Run from a fresh Python 3.11+ kernel. CPU only, no keys/GPU; 11MB official MNIST download.

All original workloads remain: 60,000 MNIST training rows for PCA, t-SNE and LDA; 10,000 Swiss-roll rows for kernel PCA/LLE; first 75 MNIST rows for the two image MDS maps; first 1,000 for all 29 MDS dimensions (1–29). The full t-SNE and MDS sweep can take several minutes. No hidden fast mode/subsample replaces them.

The 33 immutable original gist snapshots/revisions are retained below. Maintained examples repair missing plotting calls/imports, obsolete NumPy aliases and variable overwrites. Timings are this run's measurements, not universal guarantees; original screenshots remain distinct from current outputs. Interpretation findings are recorded in this notebook and the private website review, not public editorial notes.


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
               "numpy==2.3.3", "scipy==1.16.2", "scikit-learn==1.7.2", "matplotlib==3.10.9",
               "seaborn==0.13.2", "certifi==2026.10.23", "threadpoolctl==3.6.0"], check=True)
import os
import io
import ssl
import hashlib
import urllib.request
from pathlib import Path
from time import perf_counter
import numpy as np
import certifi
import matplotlib.pyplot as plt
from matplotlib import pyplot
from threadpoolctl import threadpool_limits, threadpool_info
blas_limit = threadpool_limits(limits=1, user_api="blas")
openmp_limit = threadpool_limits(limits=4, user_api="openmp")
np.set_printoptions(precision=8, suppress=True, linewidth=100)
print("Python:", sys.version.split()[0])
print("BLAS threads: 1; OpenMP threads: 4")
print("Original workloads: 60000 MNIST / 10000 Swiss-roll / 75 image maps / 1000 MDS sweep rows.")

## Example 1: gistfile1.txt

In [ ]:
from sklearn.decomposition import PCA, IncrementalPCA, KernelPCA
print("PCA and plotting imports ready.")

## Example 2: gistfile1.txt

In [ ]:
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
MNIST_SHA256 = "731c5ac602752760c8e48fbffcf8c3b850d9dc2a2aedcf2cc48468fc17b673d1"
cached_file = os.environ.get("TDB_MNIST_NPZ")
if cached_file:
    mnist_bytes = Path(cached_file).read_bytes()
else:
    with urllib.request.urlopen(MNIST_URL, context=ssl.create_default_context(cafile=certifi.where()), timeout=60) as response:
        mnist_bytes = response.read()
assert hashlib.sha256(mnist_bytes).hexdigest() == MNIST_SHA256
with np.load(io.BytesIO(mnist_bytes), allow_pickle=False) as data:
    train_X, train_y = data["x_train"], data["y_train"]
    test_X, test_y = data["x_test"], data["y_test"]
plt.figure(figsize=(4, 4))
plt.imshow(train_X[0], cmap="gray")
plt.title(f"First MNIST training image: {train_y[0]}")
plt.axis("off")
print("MNIST SHA-256:", hashlib.sha256(mnist_bytes).hexdigest())
print("First training label:", train_y[0])
plt.show()

## Example 3: gistfile1.txt

In [ ]:
print("The shape of the data", train_X.shape)
print("The shape of the test data", test_X.shape)

## Example 4: gistfile1.txt

In [ ]:
train_X_reshaped = train_X.reshape(60000, 28 * 28)
print("The shape of the reshaped data", train_X_reshaped.shape)

## Example 5: gistfile1.txt

In [ ]:
st = perf_counter()
pca_100 = PCA(n_components=100, svd_solver="full")
X_100D = pca_100.fit_transform(train_X_reshaped)
print("Execution time:", perf_counter() - st, "seconds")
print("The shape of the reduced data", X_100D.shape)

## Example 6: gistfile1.txt

In [ ]:
print(pca_100.explained_variance_ratio_)

## Example 7: gistfile1.txt

In [ ]:
cumsum = np.cumsum(pca_100.explained_variance_ratio_)
eligible = np.flatnonzero(cumsum >= 0.90)
assert eligible.size, "The retained spectrum does not reach 90%"
d = int(eligible[0] + 1)
print("The number of required components to get explained variance of 90 % is", d)

## Example 8: gistfile1.txt

In [ ]:
pca_90 = PCA(n_components=0.90, svd_solver="full")
X_reduced_90 = pca_90.fit_transform(train_X_reshaped)
print("The shape of the dimensionally reduced data is", X_reduced_90.shape)
print("Retained variance ratio:", pca_90.explained_variance_ratio_.sum())

## Example 9: gistfile1.txt

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, len(cumsum) + 1), cumsum)
plt.axhline(0.90, linestyle="--", color="gray")
plt.xlabel("Dimension")
plt.ylabel("Explained variance")
plt.title("MNIST PCA cumulative explained variance")
plt.tight_layout()
print("90% crossing dimension:", d)
plt.show()

## Example 10: gistfile1.txt

In [ ]:
X_recovered_flat = pca_90.inverse_transform(X_reduced_90)
X_recovered = X_recovered_flat.reshape(60000, 28, 28)
plt.figure(figsize=(4, 4))
plt.imshow(X_recovered[0], cmap="gray")
plt.title("MNIST reconstruction retaining 90% variance")
plt.axis("off")
reconstruction_mse = np.mean((train_X_reshaped.astype(float) - X_recovered_flat) ** 2)
print("Reconstructed shape:", X_recovered.shape)
print("Per-pixel reconstruction MSE:", reconstruction_mse)
plt.show()

## Example 11: gistfile1.txt

In [ ]:
st = perf_counter()
full_pca = PCA(n_components=100, svd_solver="full")
X_full = full_pca.fit_transform(train_X_reshaped)
print("Execution time using full SVD", perf_counter() - st, "seconds")
st = perf_counter()
rnd_pca = PCA(n_components=100, svd_solver="randomized", random_state=42)
X_randomized = rnd_pca.fit_transform(train_X_reshaped)
print("Execution time using randomized solver", perf_counter() - st, "seconds")
print("Full/randomized variance:", full_pca.explained_variance_ratio_.sum(), rnd_pca.explained_variance_ratio_.sum())

## Example 12: gistfile1.txt

In [ ]:
st = perf_counter()
n_batches = 100
inc_pca = IncrementalPCA(n_components=100)
for X_batch in np.array_split(train_X_reshaped, n_batches):
    inc_pca.partial_fit(X_batch)
X_incremental = inc_pca.transform(train_X_reshaped)
print("Execution time using Incremental PCA", perf_counter() - st, "seconds")
print("Incremental shape / samples seen:", X_incremental.shape, inc_pca.n_samples_seen_)

## Example 13: gistfile1.txt

In [ ]:
from sklearn.datasets import make_swiss_roll
swiss_X, swiss_color = make_swiss_roll(n_samples=10000, noise=0.0, random_state=42)
print("The shape of the Swiss-roll data", swiss_X.shape, swiss_color.shape)
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(swiss_X[:, 0], swiss_X[:, 1], swiss_X[:, 2], c=swiss_color, cmap="rainbow", s=2)
ax.set_title("Swiss roll in 3D")
plt.tight_layout()
plt.show()

## Example 14: gistfile1.txt

In [ ]:
kpca_rbf = KernelPCA(n_components=2, kernel="rbf", gamma=0.04, eigen_solver="arpack", random_state=42)
X_rbf = kpca_rbf.fit_transform(swiss_X)
plt.figure(figsize=(7, 5))
plt.scatter(X_rbf[:, 0], X_rbf[:, 1], c=swiss_color, cmap="rainbow", s=2)
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("Swiss roll: RBF kernel PCA, gamma=0.04")
plt.tight_layout()
print("Embedding shape:", X_rbf.shape)
print("Leading eigenvalues:", kpca_rbf.eigenvalues_)
plt.show()

## Example 15: gistfile1.txt

In [ ]:
kpca_sigmoid = KernelPCA(n_components=2, kernel="sigmoid", gamma=0.001, eigen_solver="arpack", random_state=42)
X_sigmoid = kpca_sigmoid.fit_transform(swiss_X)
plt.figure(figsize=(7, 5))
plt.scatter(X_sigmoid[:, 0], X_sigmoid[:, 1], c=swiss_color, cmap="rainbow", s=2)
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("Swiss roll: sigmoid kernel PCA, gamma=0.001")
plt.tight_layout()
print("Embedding shape:", X_sigmoid.shape)
print("Leading eigenvalues:", kpca_sigmoid.eigenvalues_)
plt.show()

## Example 16: gistfile1.txt

In [ ]:
swiss_pca = PCA(n_components=2, svd_solver="full")
X_swiss_pca = swiss_pca.fit_transform(swiss_X)
plt.figure(figsize=(7, 5))
plt.scatter(X_swiss_pca[:, 0], X_swiss_pca[:, 1], c=swiss_color, cmap="rainbow", s=2)
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("Swiss roll: ordinary PCA")
plt.tight_layout()
print("Embedding shape:", X_swiss_pca.shape)
plt.show()

## Example 17: gistfile1.txt

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding
lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10, eigen_solver="arpack", random_state=42)
X_lle = lle.fit_transform(swiss_X)
plt.figure(figsize=(7, 5))
plt.scatter(X_lle[:, 0], X_lle[:, 1], c=swiss_color, cmap="rainbow", s=2)
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("Swiss roll: LLE")
plt.tight_layout()
print("Embedding shape:", X_lle.shape)
print("LLE reconstruction objective:", lle.reconstruction_error_)
plt.show()

## Example 18: gistfile1.txt

In [ ]:
from sklearn.manifold import TSNE, trustworthiness
import seaborn as sns
import matplotlib.patheffects as PathEffects
print("t-SNE and labeled-scatter imports ready.")

## Example 19: gistfile1.txt

In [ ]:
time_start = perf_counter()
tsne = TSNE(n_components=2, perplexity=30, max_iter=1000, init="pca", learning_rate="auto",
            method="barnes_hut", angle=0.5, random_state=42, n_jobs=-1, verbose=1)
tsne_results = tsne.fit_transform(train_X_reshaped)
print("t-SNE done! Time elapsed:", perf_counter() - time_start, "seconds")
print("Embedding shape:", tsne_results.shape)
print("Final KL divergence / iterations:", tsne.kl_divergence_, tsne.n_iter_)

## Example 20: gistfile1.txt

In [ ]:
def plot_scatter(x, colors, title):
    classes = np.unique(colors)
    palette = np.array(sns.color_palette("hls", len(classes)))
    print(palette)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(x[:, 0], x[:, 1], c=palette[colors.astype(int)], s=2, alpha=0.5)
    for value in classes:
        xtext, ytext = np.median(x[colors == value], axis=0)
        label = ax.text(xtext, ytext, str(value), fontsize=18)
        label.set_path_effects([PathEffects.Stroke(linewidth=4, foreground="white"), PathEffects.Normal()])
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="datalim")
    fig.tight_layout()
    return fig, ax
plot_scatter(tsne_results, train_y, "MNIST: full 60000-row t-SNE embedding")
plt.show()

## Example 21: gistfile1.txt

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
st = perf_counter()
lda_3 = LinearDiscriminantAnalysis(n_components=3, solver="svd")
train_lda = lda_3.fit_transform(train_X_reshaped, train_y)
print("Execution time:", perf_counter() - st, "seconds")
print("The shape of the reduced data", train_lda.shape)

## Example 22: gistfile1.txt

In [ ]:
print("The explained discriminant variance ratio:", lda_3.explained_variance_ratio_.sum())

## Example 23: gistfile1.txt

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(train_lda[:, 0], train_lda[:, 1], train_lda[:, 2], c=train_y, cmap="rainbow", s=1, alpha=0.3)
ax.set_title("The 3D LDA-reduced MNIST data")
plt.tight_layout()
print("Plotted MNIST rows:", len(train_lda))
plt.show()

## Example 24: gistfile1.txt

In [ ]:
st = perf_counter()
lda_2 = LinearDiscriminantAnalysis(n_components=2, solver="svd")
train_lda_2D = lda_2.fit_transform(train_X_reshaped, train_y)
print("Execution time:", perf_counter() - st, "seconds")
print("The shape of the reduced data", train_lda_2D.shape)
print("The explained discriminant variance ratio:", lda_2.explained_variance_ratio_.sum())

## Example 25: gistfile1.txt

In [ ]:
plot_scatter(train_lda_2D, train_y, "2D MNIST-reduced data using LDA")
plt.show()

## Example 26: gistfile1.txt

In [ ]:
from sklearn.manifold import MDS
from sklearn.metrics.pairwise import manhattan_distances, euclidean_distances
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
def make_mds(n_components=2, dissimilarity="euclidean"):
    return MDS(n_components=n_components, metric=True, dissimilarity=dissimilarity,
               random_state=0, n_init=4, max_iter=300, eps=1e-3, normalized_stress=False, n_jobs=1)
print("MDS, distance metrics and image annotations ready.")

## Example 27: gistfile1.txt

In [ ]:
toy_X = np.array([[0, 0, 0], [0, 0, 1], [1, 1, 1], [0, 1, 0], [0, 1, 1]])
toy_mds = make_mds()
X_transform = toy_mds.fit_transform(toy_X)
print(X_transform)

## Example 28: gistfile1.txt

In [ ]:
print("The stress factor is", toy_mds.stress_)

## Example 29: gistfile1.txt

In [ ]:
colors = ["r", "g", "b", "c", "m"]
fig = plt.figure(figsize=(12, 5))
ax = fig.add_subplot(121, projection="3d")
ax.scatter(toy_X[:, 0], toy_X[:, 1], toy_X[:, 2], s=64, c=colors)
ax.set_title("Original points")
ax = fig.add_subplot(122)
ax.scatter(X_transform[:, 0], X_transform[:, 1], s=64, c=colors)
ax.set_title("Embedding in 2D")
fig.tight_layout()
print("Toy points / embedding:", toy_X.shape, X_transform.shape)
plt.show()

## Example 30: gistfile1.txt

In [ ]:
def mapData(dist_matrix, images, labels, title):
    model = make_mds(dissimilarity="precomputed")
    pts = model.fit_transform(dist_matrix)
    print("The stress factor is", model.stress_)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.scatterplot(x=pts[:, 0], y=pts[:, 1], hue=labels.astype(str), palette="colorblind", ax=axes[0])
    axes[0].set_title("Embedding in 2D")
    axes[1].scatter(pts[:, 0], pts[:, 1], s=2)
    for image, position in zip(images, pts):
        box = OffsetImage(image.reshape(28, 28), zoom=0.5, cmap="gray")
        axes[1].add_artist(AnnotationBbox(box, position, frameon=False))
    axes[1].set_title(title)
    fig.tight_layout()
    plt.show()
    return model, pts
print("MDS digit-image mapping function ready.")

## Example 31: gistfile1.txt

In [ ]:
dist_euclid_75 = euclidean_distances(train_X_reshaped[:75].astype(float))
mds_euclid, pts_euclid = mapData(dist_euclid_75, train_X_reshaped[:75], train_y[:75], "Metric MDS with Euclidean distance")
print("Mapped rows:", len(pts_euclid))

## Example 32: gistfile1.txt

In [ ]:
dist_L1_75 = manhattan_distances(train_X_reshaped[:75].astype(float))
mds_manhattan, pts_manhattan = mapData(dist_L1_75, train_X_reshaped[:75], train_y[:75], "Metric MDS with Manhattan distance")
print("Mapped rows:", len(pts_manhattan))

## Example 33: gistfile1.txt

In [ ]:
dist_euclid_1000 = euclidean_distances(train_X_reshaped[:1000].astype(float))
stress = []
sweep_models = []
max_range = 30
for dim in range(1, max_range):
    model = make_mds(n_components=dim, dissimilarity="precomputed")
    model.fit_transform(dist_euclid_1000)
    stress.append(model.stress_)
    sweep_models.append(model)
    print(f"MDS components={dim:2d}, stress={model.stress_:.6f}, iterations={model.n_iter_}", flush=True)
plt.figure(figsize=(9, 5))
plt.plot(range(1, max_range), stress, marker=".")
plt.xticks(range(1, max_range, 2))
plt.xlabel("n_components")
plt.ylabel("Raw stress")
plt.title("MNIST: MDS stress on the original first 1000 rows")
plt.tight_layout()
plt.show()

## Independent checks and interpretation

PCA variance/reconstruction are checked against independent covariance eigenvalues, transforms and residual-energy identities. Both kernel PCA eigen-equations are checked blockwise on all 10,000 rows, avoiding a second resident 800MB kernel. LLE neighbor weights/objective are independently rebuilt. Full t-SNE is checked for finite shape/objective plus independently computed trustworthiness on a fixed 1,000-row diagnostic sample (not a substituted fit). LDA transforms and between/within-class discriminant-energy ratios are checked. Every MDS raw stress, including all 29 sweep fits, is independently recomputed.


In [ ]:
from scipy.spatial.distance import cdist
from sklearn.neighbors import NearestNeighbors
assert train_X.shape == (60000, 28, 28) and test_X.shape == (10000, 28, 28)
assert train_X_reshaped.shape == (60000, 784) and swiss_X.shape == (10000, 3)
assert X_100D.shape == X_randomized.shape == X_incremental.shape == (60000, 100)
assert inc_pca.n_samples_seen_ == 60000 and d == 87 and X_reduced_90.shape == (60000, 87)
# Independent full covariance spectrum and residual-energy reconstruction identity.
covariance = np.cov(train_X_reshaped.astype(float), rowvar=False)
eigenvalues = np.linalg.eigvalsh(covariance)[::-1]
np.testing.assert_allclose(pca_100.explained_variance_, eigenvalues[:100], rtol=1e-9, atol=1e-7)
np.testing.assert_allclose(pca_100.explained_variance_ratio_, eigenvalues[:100] / eigenvalues.sum(), rtol=1e-9)
for model, reduced in [(pca_100, X_100D), (pca_90, X_reduced_90), (full_pca, X_full), (rnd_pca, X_randomized), (inc_pca, X_incremental)]:
    np.testing.assert_allclose(model.components_ @ model.components_.T, np.eye(len(model.components_)), atol=1e-9)
    projected = (train_X_reshaped[:256] - model.mean_) @ model.components_.T
    if model is rnd_pca:
        assert np.linalg.norm(projected - reduced[:256]) / np.linalg.norm(projected) < 0.02
    else:
        np.testing.assert_allclose(projected, reduced[:256], rtol=1e-5, atol=1e-7)
np.testing.assert_allclose(reconstruction_mse, eigenvalues[87:].sum() * 59999 / 60000 / 784, rtol=1e-9)
# Kernel PCA eigen-equations and independent centering, with 200-row blocks.
for model, embedding, kernel in [(kpca_rbf, X_rbf, "rbf"), (kpca_sigmoid, X_sigmoid, "sigmoid")]:
    assert embedding.shape == (10000, 2) and np.isfinite(embedding).all()
    vectors = model.eigenvectors_; values = model.eigenvalues_
    means = model._centerer.K_fit_rows_; grand_mean = model._centerer.K_fit_all_
    for start in range(0, 10000, 200):
        rows = swiss_X[start:start+200]
        kernel_block = np.exp(-0.04 * cdist(rows, swiss_X, "sqeuclidean")) if kernel == "rbf" else np.tanh(0.001 * (rows @ swiss_X.T) + 1)
        row_means = kernel_block.mean(axis=1)
        np.testing.assert_allclose(row_means, means[start:start+len(rows)], atol=1e-12)
        centered = kernel_block - row_means[:, None] - means[None, :] + grand_mean
        np.testing.assert_allclose(centered @ vectors, vectors[start:start+len(rows)] * values, rtol=1e-4, atol=1e-6)
    np.testing.assert_allclose(embedding, vectors * np.sqrt(values), atol=1e-7)
# Independently reconstruct LLE's regularized local weights for all 10000 rows.
neighbors = NearestNeighbors(n_neighbors=11).fit(swiss_X).kneighbors(swiss_X, return_distance=False)[:, 1:]
lle_objective = 0.0
for row, near in enumerate(neighbors):
    offsets = swiss_X[near] - swiss_X[row]
    gram = offsets @ offsets.T
    gram.flat[::11] += 1e-3 * np.trace(gram)
    weights = np.linalg.solve(gram, np.ones(10)); weights /= weights.sum()
    lle_objective += np.sum((X_lle[row] - weights @ X_lle[near]) ** 2)
np.testing.assert_allclose(lle.reconstruction_error_, lle_objective, rtol=1e-4, atol=1e-9)
assert tsne_results.shape == (60000, 2) and np.isfinite(tsne_results).all()
assert np.isfinite(tsne.kl_divergence_) and tsne.kl_divergence_ > 0 and tsne.n_iter_ <= 1000
# Fixed diagnostic sample only; the t-SNE fit above still uses all 60000 rows.
probe = np.linspace(0, 59999, 1000, dtype=int)
high_dist = cdist(train_X_reshaped[probe].astype(float), train_X_reshaped[probe].astype(float))
np.fill_diagonal(high_dist, np.inf)
rank_order = np.argsort(high_dist, axis=1)
ranks = np.empty_like(rank_order)
ranks[np.arange(1000)[:, None], rank_order] = np.arange(1, 1001)
low_dist = cdist(tsne_results[probe], tsne_results[probe]); np.fill_diagonal(low_dist, np.inf)
near_low = np.argsort(low_dist, axis=1)[:, :10]
penalty = np.maximum(ranks[np.arange(1000)[:, None], near_low] - 10, 0).sum()
manual_trust = 1 - 2 * penalty / (1000 * 10 * (2000 - 30 - 1))
trust = trustworthiness(train_X_reshaped[probe], tsne_results[probe], n_neighbors=10)
np.testing.assert_allclose(trust, manual_trust, atol=1e-12)
assert trust > 0.8
# LDA transforms; independent discriminant ratios on all rows, not PCA variance.
for model, transformed in [(lda_3, train_lda), (lda_2, train_lda_2D)]:
    np.testing.assert_allclose((train_X_reshaped[:256] - model.xbar_) @ model.scalings_[:, :model.n_components], transformed[:256], atol=1e-8)
    all_projection = (train_X_reshaped.astype(float) - model.xbar_) @ model.scalings_
    class_means = np.array([all_projection[train_y == label].mean(axis=0) for label in range(10)])
    counts = np.bincount(train_y)
    between = np.sum(counts[:, None] * class_means ** 2, axis=0)
    np.testing.assert_allclose(model.explained_variance_ratio_, (between / between.sum())[:model.n_components], rtol=1e-8, atol=1e-10)
    assert np.isfinite(transformed).all()
# Independently recompute raw stress for toy data, both 75-row maps and all29 sweep fits.
mds_checks = [(toy_mds, cdist(toy_X, toy_X)), (mds_euclid, dist_euclid_75), (mds_manhattan, dist_L1_75)]
mds_checks += [(model, dist_euclid_1000) for model in sweep_models]
for model, distances in mds_checks:
    embedded_distances = cdist(model.embedding_, model.embedding_)
    raw_stress = np.sum(np.triu((distances - embedded_distances) ** 2, k=1))
    np.testing.assert_allclose(model.stress_, raw_stress, rtol=1e-9, atol=1e-9)
    assert np.isfinite(model.embedding_).all() and model.n_init == 4
np.testing.assert_allclose(dist_euclid_75, cdist(train_X_reshaped[:75].astype(float), train_X_reshaped[:75].astype(float)), atol=1e-8)
np.testing.assert_allclose(dist_L1_75, cdist(train_X_reshaped[:75].astype(float), train_X_reshaped[:75].astype(float), "cityblock"), atol=1e-8)
assert len(sweep_models) == 29 and len(stress) == 29
print("All33 examples retain the original full workloads and produce finite outputs.")
print("PCA spectrum/transforms/reconstruction, both full kernel eigen-equations and 10000 LLE local weights pass.")
print("Full60000-row t-SNE KL:", tsne.kl_divergence_, "; fixed1000-row diagnostic trustworthiness:", trust)
print("Both full-data LDA transforms/discriminant ratios and all32 independent MDS raw stresses pass.")
blas_limit.restore_original_limits(); openmp_limit.restore_original_limits()

## Maintenance and scientific scope

- Read the same official Keras MNIST NPZ directly via NumPy/allow_pickle=False, checksum-verified, instead of adding TensorFlow just for loading data. MNIST: Yann LeCun/Corinna Cortes, CC BY-SA3.0; [source](https://keras.io/api/datasets/mnist/).
- Use explicit seeds, monotonic timers and bounded numerical threading. Preserve all rows, 100 IPCA batches, kernel gamma values, LLE10neighbors, t-SNEperplexity30/1000iterations, and MDS4initializations/300iterations/29dimensions. Current t-SNE PCA initialization and auto learning rate are explicit version-maintenance choices, not old-run equivalence.
- Keep PCA estimators/data snapshots distinct: source overwrites would lose the 90%-variance projection before reconstruction and later replace Swiss-roll data with toy points. Supply missing plt/imports, show() calls, t-SNE plot invocation and third plot_scatter title argument. Remove obsolete np.int and conflicting color/cmap usage; cast pixel data to float for distance arithmetic.
- Kernel PCA/LLE plots can rotate/flip; sigmoid kernels need not be positive semidefinite and this example does not establish a globally valid Mercer kernel. Check the retained positive eigenspace. Approximate solvers/t-SNE do not guarantee global distance preservation or causal class separation.
- PCA explained data variance and LDA explained discriminant variance are different quantities, not directly comparable percentages. LDA uses class labels. t-SNE is primarily visualization, not an out-of-sample classifier; visible clusters are not classification evidence.
- The source's million-dimensional mean-distance expression is missing a square root: approximately sqrt(1000000/6)=408.25, not 1000000/6. Source speedup/structure-retention claims are not universal.
- IPCA here keeps the input resident; reduced feature count does not itself prove lower peak memory. Timings include the shown scopes only. MDS is local stress minimization; independent fits need not have strictly monotone stress. Seven of784features is about0.893% retained (99.107% fewer), not a0.8% reduction. Stress/elbow and class maps do not prove information preservation or an optimal universal dimension.
- Original historical prose/screenshots remain in the website under the user's source-preservation rule; interpretation findings stay in this notebook/private review, not public editorial caveats.


## Immutable original gist appendix (not executable)

### f4a4e45d6ba31b629300939f955e55a1 / 4ed5dff4bb4806b85951a09f01a540a4f2488de1

Original filename: gistfile1.txt

```python
from sklearn.decomposition import PCA
from keras.datasets import mnist
from matplotlib import pyplot
import numpy as np
import time
```

### 0456e936ebe1fc42c356b94b5fd99afe / e13f7107591894bb1bdd18fc66c60df498541b6a

Original filename: gistfile1.txt

```python
# Loading and Plotting random image of the mnsit dataset
(train_X, train_y), (test_X, test_y) = mnist.load_data()

# plotting the first image of the MNSIt dataset
pyplot.imshow(train_X[0], cmap=pyplot.get_cmap('gray'))
pyplot.show()
```

### 578744c713fd87d3fff5688af37ea6ee / 3a1b1b03f8e675db408133bc566175a794c0efb5

Original filename: gistfile1.txt

```python
print('The shape of the data',train_X.shape)
```

### e5613646b0e1ef5f1e8a8214198c34e9 / 940c53ee4b64011de8199d33018de456c44ad816

Original filename: gistfile1.txt

```python
train_X_reshaped = train_X.reshape(60000, 28*28)
print('The shape of the reshaped data', train_X_reshaped.shape)
```

### fd9439a668473e426b21b2b6ab1d50b8 / 11987474ac18c2c3289b275fe8aa89bdcfc13fc8

Original filename: gistfile1.txt

```python
# Apply PCA to MNSIT dataset

# get the start time
st = time.time()

pca = PCA(n_components = 100)
X_100D = pca.fit_transform(train_X_reshaped)

# get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Execution time:', elapsed_time, 'seconds')
print('The shape of the reduced data',X_100D.shape)
```

### d5923aaf9025f6806ede71f3b739b565 / e4de4598bc453feabb8270c5d1a745f910f56946

Original filename: gistfile1.txt

```python
# Calcualte the explain varinace ration for each dimesnion
pca.explained_variance_ratio_
```

### a3ada0c439a900f3f15cff99f8a34315 / f04db14fd25eb4d6917cda46e1fa27b11bec2819

Original filename: gistfile1.txt

```python
# Choosing the Right Number of Dimensions
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.90) + 1 
print ('The number of required components to get explained variance of 90 % is',d)
```

### 67fb2ab40fb0c4fa5aca5496f88ed75e / d5c9aa8fd570931ff4162456d2eecaaaad52e6cb

Original filename: gistfile1.txt

```python
# Defining the varinace explained ratio
pca = PCA(n_components=0.90)
X_reduced = pca.fit_transform(train_X_reshaped)
print('The shape of the dimesnialy reduced data is',X_reduced.shape)
```

### 495e4274dd5c87be5a3c3bcd4deb80dd / 388f3091041cbf2b9214e6af5a9f5a5e30ea9b10

Original filename: gistfile1.txt

```python
#  Plotting the explained variance as a function of the number of dimensions
plt.style.use('ggplot')
plt.figure(figsize=(10,10))

pyplot.plot(np.arange(1,len(cumsum)+1), cumsum)
pyplot.xlabel('Dimension')
pyplot.ylabel('Explained Varinace')
```

### 2f66b6a1e57543c4a5363eb669e3e752 / 77c6c91888ba9aa00f22d3c835b10229f3df44fd

Original filename: gistfile1.txt

```python
# decompressing the dimensioality reduced images 
X_recovered = pca.inverse_transform(X_reduced)
X_recovered = X_recovered.reshape(60000,28,28)

# Plotting the decompressed image
plt.style.use('classic')
pyplot.imshow(X_recovered[0], cmap=pyplot.get_cmap('gray'))
pyplot.show()
```

### ce6114c0d1fa9860e6231f6782c4f886 / c3b803e017ad813590f38ed2ee9afeddaed7c616

Original filename: gistfile1.txt

```python
# Calculate the execution time for the PCA using full SVD solver

# get the start time
st = time.time()

rnd_pca = PCA(n_components=100, svd_solver="full")
X_reduced_rand = rnd_pca.fit_transform(train_X_reshaped)

# get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Execution time using full SVD', elapsed_time, 'seconds')

# Calculate the execution time for the PCA using randomized solver

# get the start time
st = time.time()

rnd_pca = PCA(n_components=100, svd_solver="randomized")
X_reduced_rand = rnd_pca.fit_transform(train_X_reshaped)

# get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Execution time using randomized solver', elapsed_time, 'seconds')
```

### ea50b4fc1069cee64b38770adc2ccf4f / f5abbe4aa98434bdaf1ea9602ee5c210c9b7e6c8

Original filename: gistfile1.txt

```python
# get the start time
st = time.time()    

from sklearn.decomposition import IncrementalPCA
n_batches = 100 # number of batches 
inc_pca = IncrementalPCA(n_components=100) # Define the Incremental PCA
for X_batch in np.array_split(train_X_reshaped, n_batches): # Apply PCA on each batch 
    inc_pca.partial_fit(X_batch)
    
X_reduced = inc_pca.transform(train_X_reshaped)

# get the end time
et = time.time()
# get the execution time
elapsed_time = et - st
print('Execution time using Incremental PCA', elapsed_time, 'seconds')
```

### c4566ec1ec25897e5e6b74ed11897fcc / f16d7242e58301a21958aecc3584d966283818a5

Original filename: gistfile1.txt

```python
# create and plot swiss roll 
from sklearn import datasets
from mpl_toolkits.mplot3d import Axes3D
from sklearn.datasets import make_swiss_roll

# generate 3D swiss roll data
X, y = datasets.make_swiss_roll(n_samples=10000, noise=0.0, random_state=None)
print('The shape of the swiss rol1 data ',X.shape, y.shape)

# plot the swiss roll data 
fig = plt.figure(figsize=(7,7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=y, cmap=plt.cm.rainbow)
plt.title('Swiss Roll in 3D')
plt.show()
```

### 9b5cd58f4b221c76da6d47adadbcf8dc / e3df033a852dfe770279a96ee7e59649c899925d

Original filename: gistfile1.txt

```python
from sklearn.decomposition import KernelPCA
rbf_pca = KernelPCA(n_components = 2, kernel="rbf", gamma=0.04)
X_reduced_rbf = rbf_pca.fit_transform(X)
X_reduced_rbf.shape

plt.scatter(X_reduced_rbf[:, 0],X_reduced_rbf[:,1], c=y, cmap=plt.cm.rainbow )
plt.xlabel("X") 
plt.ylabel("y")
plt.title("Reduced swiss roll with RBF PCA Kernel with gama = 0.04 ")
plt.show()
```

### 0f7dfba729dc88093c0a8275bcb10974 / beecfdb4cf2377479c685234554f2286c240386a

Original filename: gistfile1.txt

```python
rbf_pca = KernelPCA(n_components = 2, kernel="sigmoid", gamma=0.001)
X_reduced_rbf = rbf_pca.fit_transform(X)
X_reduced_rbf.shape

plt.scatter(X_reduced_rbf[:, 0],X_reduced_rbf[:,1], c=y, cmap=plt.cm.rainbow )
plt.xlabel("X") 
plt.ylabel("y")
plt.show()
```

### 74210d4c4a07b0910ace0483ecf30a95 / 7add21af0fc90389024a890ecccbcc8fb908fe18

Original filename: gistfile1.txt

```python
pca = PCA(n_components = 2)
X_reduced_pca = pca.fit_transform(X)

plt.scatter(X_reduced_pca[:, 0],X_reduced_pca[:,1], c=y, cmap=plt.cm.rainbow )
plt.xlabel("X") 
plt.ylabel("y")
plt.title(" Reduced swiss roll with PCA ")
plt.show()
```

### fe8a4d52a586e5fd1ad6beaf940ffe01 / 71114ad2dff11e368888d592e6c6285af89c6b8f

Original filename: gistfile1.txt

```python
from sklearn.manifold import LocallyLinearEmbedding

lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10)
X_reduced_LLE = lle.fit_transform(X)

plt.scatter(X_reduced_LLE[:, 0],X_reduced_LLE[:,1], c=y, cmap=plt.cm.rainbow )
plt.xlabel("X") 
plt.ylabel("y")
plt.title(" Reduced swiss roll with LLE ")
plt.show()
```

### 67a37a91886939ed8ec349287f5f40c7 / 1edc5e09e9536f86c70858a9566d24a0331bd30b

Original filename: gistfile1.txt

```python
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import matplotlib.patheffects as PathEffects
%matplotlib inline
```

### ec898b67534b485cf12db1240020394a / e25ca3aec531fe53b41c61aebb106ddf58db88b8

Original filename: gistfile1.txt

```python
time_start = time.time()

tsne = TSNE()
tsne_results = tsne.fit_transform(train_X_reshaped)

print ('t-SNE done! Time elapsed: {} seconds'.format(time.time()-time_start))
```

### 19f8ae38bcd4648c51cc365673a6817b / 91c289dd5f28287b0b93c9f8815bcc96922f27a0

Original filename: gistfile1.txt

```python
def plot_scatter(x, colors):
    # choose a color palette with seaborn.
    num_classes = len(np.unique(colors))
    palette = np.array(sns.color_palette("hls", num_classes))
    print(palette)
    
    # create a scatter plot.
    f = plt.figure(figsize=(8, 8))
    ax = plt.subplot(aspect='equal')
    sc = ax.scatter(x[:,0], x[:,1],  c=palette[colors.astype(np.int)], cmap=plt.cm.get_cmap('Paired'))
    plt.xlim(-25, 25)
    plt.ylim(-25, 25)
    ax.axis('off')
    ax.axis('tight')
    # add the labels for each digit corresponding to the label
    txts = []
    
    for i in range(num_classes):
        # Position of each label at median of data points.
        xtext, ytext = np.median(x[colors == i, :], axis=0)
        txt = ax.text(xtext, ytext, str(i), fontsize=24)
        txt.set_path_effects([PathEffects.Stroke(linewidth=5, foreground="w"),
                              PathEffects.Normal()])
        txts.append(txt)

    return f, ax, sc, txts
```

### 5c137e018ddb4fb27fd1cc4c2506b145 / eb6800ad373c0565bdf908259337759c3730ccba

Original filename: gistfile1.txt

```python
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Apply PCA to MNSIT dataset

# get the start time
st = time.time()

lda = LinearDiscriminantAnalysis(n_components=3)
train_lda = lda.fit_transform(train_X_reshaped, train_y)

# get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Execution time:', elapsed_time, 'seconds')
print('The shape of the reduced data',train_lda.shape)
```

### 729fd44d9a66b4e1c74d83b4d373a06b / b1af6ab3b4b7eebcfc6b6f15d6b1247f6a4e4673

Original filename: gistfile1.txt

```python
print("The explained variance ratio: ", lda.explained_variance_ratio_.sum())
```

### 75c07b53b2a8011358f8cc8c2cb4ec84 / d26924c06ee182d9641895b5a2a31c85d0373798

Original filename: gistfile1.txt

```python
# plot the 3D LDA reduced data 
fig = plt.figure(figsize=(7,7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(train_lda[:, 0], train_lda[:, 1], train_lda[:, 2], c=train_y, cmap=plt.cm.rainbow)
plt.title('The 3D LDA reduced MNSIT data')
plt.show()
```

### 23c4c7dc2e7e6b65c6b09a16d0a9ca02 / fade9abf0e98700e45c021e538e73bad7249b388

Original filename: gistfile1.txt

```python
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Apply PCA to MNSIT dataset

# get the start time
st = time.time()

lda = LinearDiscriminantAnalysis(n_components=2)
train_lda_2D = lda.fit_transform(train_X_reshaped, train_y)

# get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Execution time:', elapsed_time, 'seconds')
print('The shape of the reduced data',train_lda_2D.shape)
print("The explained variance ratio: ", lda.explained_variance_ratio_.sum())
```

### 1d3d33442e57367e865d525a728d1705 / 2f47a417045e083fa08e2122dd7931c95457b6c4

Original filename: gistfile1.txt

```python
plot_scatter( train_lda_2D, train_y,'2D MNSIT reduced data using LDA')
```

### b6204b6e6c4f91a19c099d2eee46e98f / 2aaac7a7eeab028e0134747a148e63254f48db49

Original filename: gistfile1.txt

```python
from sklearn.manifold import MDS
from sklearn.metrics.pairwise import manhattan_distances, euclidean_distances
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
```

### d28dce10928b29ea9ebded69d7a60a01 / 106c3497eb0601e89af22eb25bbff86d39fb4090

Original filename: gistfile1.txt

```python
X = np.array([[0, 0, 0], [0, 0, 1], [1, 1, 1], [0, 1, 0], [0, 1, 1]])
mds = MDS(random_state=0)
X_transform = mds.fit_transform(X)
print(X_transform)
```

### 166b30a853b52d239b32787e70642bd9 / 62d0416f4720b390d18378a605f59d8c2c436293

Original filename: gistfile1.txt

```python
print('The stree factor is',mds.stress_)
```

### 95bd5d8fa22424d01e239d98f32a82a3 / 7181e32447f61980606f9ba27668251b6aee1bcf

Original filename: gistfile1.txt

```python
colors = ['r', 'g', 'b', 'c', 'm']
size = [64, 64, 64, 64, 64]

fig = plt.figure(2, (20,10))
ax = fig.add_subplot(121, projection='3d')
plt.scatter(X[:,0], X[:,1], zs=X[:,2], s=size, c=colors)
plt.title('Original Points')

ax = fig.add_subplot(122)
plt.scatter(X_transform[:,0], X_transform[:,1], s=size, c=colors)
plt.title('Embedding in 2D')
fig.subplots_adjust(wspace=.4, hspace=0.5)
plt.show()
```

### 5c40408f84b2d63c0f726f4a5fe61eb4 / 2333bde989cd597e4e37a28bda29d2bc3aad98f0

Original filename: gistfile1.txt

```python
def mapData(dist_matrix, X, y, metric, title):
    mds = MDS(metric=metric, dissimilarity='precomputed', random_state=0)
    # Get the embeddings
    pts = mds.fit_transform(dist_matrix)
    print('The stress factor is',mds.stress_)
    # Plot the embedding, colored according to the class of the points
    fig = plt.figure(2, (15,6))
    ax = fig.add_subplot(1,2,1)    
    ax = sns.scatterplot(x=pts[:, 0], y=pts[:, 1],
                         hue=y, palette='colorblind' )
    plt.title('Embedding in 2D')
  
    # Add the second plot
    ax = fig.add_subplot(1,2,2)
    # Plot the points again
    plt.scatter(pts[:, 0], pts[:, 1])
    
    # Annotate each point by its corresponding face image
    for x, ind in zip(X, range(pts.shape[0])):
        im = x.reshape(28,28)
        imagebox = OffsetImage(im, zoom=0.5, cmap=plt.cm.gray)
        i = pts[ind, 0]
        j = pts[ind, 1]
        ab = AnnotationBbox(imagebox, (i, j), frameon=False)
        ax.add_artist(ab)
    plt.title(title)    
    plt.show()
```

### 2b0738999a01f8ad0ce1885a59aae3d8 / 61b1efb9b820167452c4cb1966bda1e732e0dbd3

Original filename: gistfile1.txt

```python
dist_euclid = euclidean_distances(train_X_reshaped[0:75])
mapData(dist_euclid, train_X_reshaped[0:75], train_y[0:75], True, 
        'Metric MDS with Euclidean')
```

### 5b6ad3a3af619d97158d48eef81a957e / f6bca3b67f8a8d7b412675d16eca3ed08826a696

Original filename: gistfile1.txt

```python
dist_L1 = manhattan_distances(train_X_reshaped[0:75])
mapData(dist_L1, train_X_reshaped[0:75], train_y[0:75], True, 
        'Metric MDS with Manhattan')
```

### 224160960e79586f8342be98394da315 / 516d232b1fa913a1eb2de6bc9843e3b8ec4ed274

Original filename: gistfile1.txt

```python
dist_euclid = euclidean_distances(train_X_reshaped[0:1000])

stress = []
# Max value for n_components
max_range = 30
for dim in range(1, max_range):
    # Set up the MDS object
    mds = MDS(n_components=dim, dissimilarity='precomputed', random_state=0)
    # Apply MDS
    pts = mds.fit_transform(dist_euclid)
    # Retrieve the stress value
    stress.append(mds.stress_)
# Plot stress vs. n_components    
plt.plot(range(1, max_range), stress)
plt.xticks(range(1, max_range, 2))
plt.xlabel('n_components')
plt.ylabel('stress')
plt.show()
```

